In [46]:
# ==============================
# 🔧 INSTALAÇÃO
# ==============================
!pip install -q sentence-transformers bert-score transformers

# ==============================
# 📌 IMPORTS
# ==============================
from sentence_transformers import SentenceTransformer, util
from bert_score import score as bertscore
from transformers import pipeline
import re
import torch

# ==============================
# ✍️ INPUT
# ==============================
GABARITO = "o réu deve ser absolvido pela lei 2010"
RESPOSTA = "o réu deve ser absolvido"
LEGISLACAO = "2010"

# ==============================
# 🧠 MODELOS
# ==============================
sbert_model = SentenceTransformer('stjiris/bert-large-portuguese-cased-legal-mlm-sts-v1.0')

device = 0 if torch.cuda.is_available() else -1
nli_model = pipeline(
    "text-classification",
    model="MoritzLaurer/mDeBERTa-v3-base-mnli-xnli",
    device=device
)

# ==============================
# 🔍 1. BERTSCORE
# ==============================
P, R, F1 = bertscore([RESPOSTA], [GABARITO], lang="pt", verbose=False)

bertscore_result = {
    "precision": P.item(),
    "recall": R.item(),
    "f1": F1.item()
}

# ==============================
# 🔍 2. SBERTSCORE
# ==============================
def segmentar_sentencas(texto):
    return [s.strip() for s in texto.split('.') if s.strip()]

resp_sents = segmentar_sentencas(RESPOSTA)
gab_sents = segmentar_sentencas(GABARITO)

emb_resp = sbert_model.encode(resp_sents, convert_to_tensor=True)
emb_gab = sbert_model.encode(gab_sents, convert_to_tensor=True)

cos_sim = util.cos_sim(emb_resp, emb_gab)

precision = cos_sim.max(dim=1).values.mean().item()
recall = cos_sim.max(dim=0).values.mean().item()
f1 = 2 * (precision * recall) / (precision + recall)

sbertscore_result = {
    "precision": precision,
    "recall": recall,
    "f1": f1
}

# ==============================
# 🔍 3. MATCH LEGISLAÇÃO (ROBUSTO)
# ==============================

def normalizar_numero(num):
    """Remove pontos e zeros à esquerda"""
    return re.sub(r'\D', '', num).lstrip('0')

def extrair_referencias_legais(texto):
    """
    Captura padrões como:
    - Lei 8.112
    - Lei nº 8.112/90
    - 8112
    - 8.112
    """
    padroes = [
        r'lei\s*(?:n[ºo]\s*)?(\d{1,5}(?:\.\d{3})*(?:/\d{2,4})?)',
        r'\b(\d{1,5}(?:\.\d{3})*(?:/\d{2,4})?)\b'
    ]

    encontrados = []

    for padrao in padroes:
        matches = re.findall(padrao, texto.lower())
        encontrados.extend(matches)

    # Normaliza tudo (8112, 8.112 → 8112)
    return [normalizar_numero(n) for n in encontrados]

def verificar_legislacao(resposta, legislacao_esperada):
    if not legislacao_esperada:
        return {
            "esperada": None,
            "encontrados": [],
            "match": None
        }

    esperado_norm = normalizar_numero(legislacao_esperada)
    encontrados = extrair_referencias_legais(resposta)

    match = esperado_norm in encontrados

    return {
        "esperada": legislacao_esperada,
        "esperada_norm": esperado_norm,
        "encontrados": encontrados,
        "match": match
    }

legislacao_result = verificar_legislacao(RESPOSTA, LEGISLACAO)

# ==============================
# 🔍 4. NLI
# ==============================
def calcular_nli(resposta, gabarito):
    result = nli_model(f"{gabarito} </s> {resposta}")

    label = result[0]['label'].upper()
    score = result[0]['score']

    if label == "ENTAILMENT":
        nli_score = score
    elif label == "NEUTRAL":
        nli_score = 0.5 * score
    elif label == "CONTRADICTION":
        nli_score = 1 - score

    return {
        "label": label,
        "score_bruto": score,
        "score_convertido": nli_score
    }

nli_result = calcular_nli(RESPOSTA, GABARITO)

# ==============================
# 🎯 5. NOTA FINAL (AJUSTADA)
# ==============================
def calcular_nota_final(
    sbert_f1,
    nli_score,
    match_legislacao,
    valor_questao=1
):
    # 🔻 Contradição forte
    if nli_score < 0.2:
        return round(nli_score * valor_questao, 2)

    # 🔻 Score base
    score_base = (0.7 * sbert_f1) + (0.3 * nli_score)

    # 🔻 Penalização só se aplicável
    if match_legislacao is False:
        score_base *= 0.7

    return round(score_base * valor_questao, 2)

final_result = calcular_nota_final(
    sbert_f1=sbertscore_result["f1"],
    nli_score=nli_result["score_convertido"],
    match_legislacao=legislacao_result["match"],
    valor_questao=1
)

# ==============================
# 📊 RESULTADOS
# ==============================
print("\n==============================")
print("📊 RESULTADOS")
print("==============================\n")

print("🔹 BERTScore:")
print(bertscore_result)

print("\n🔹 SBERTScore:")
print(sbertscore_result)

print("\n🔹 Legislação:")
print(legislacao_result)

print("\n🔹 NLI:")
print(nli_result)

print("\n🔹 Nota Final:")
print(str(final_result) + " / 1.0")


# ==============================
# 💬 6. EXPLICAÇÃO DA NOTA
# ==============================
def gerar_explicacao(sbert_f1, nli_result, legislacao_result, nota_final, valor_questao=1):
    explicacao = []

    # --- NLI ---
    label = nli_result["label"]
    nli_score = nli_result["score_convertido"]

    if label == "CONTRADICTION":
        explicacao.append(f"❌ A resposta contradiz o gabarito (NLI: contradição com score {nli_result['score_bruto']:.2f}), o que resultou em penalização severa.")
    elif label == "ENTAILMENT":
        explicacao.append(f"✅ A resposta está alinhada com o gabarito (NLI: implicação com score {nli_result['score_bruto']:.2f}).")
    elif label == "NEUTRAL":
        explicacao.append(f"⚠️ A resposta é parcialmente compatível com o gabarito (NLI: neutro com score {nli_result['score_bruto']:.2f}).")

    # --- SBERT ---
    if sbert_f1 >= 0.85:
        explicacao.append(f"✅ Alta similaridade semântica com o gabarito (SBERTScore F1: {sbert_f1:.2f}).")
    elif sbert_f1 >= 0.65:
        explicacao.append(f"⚠️ Similaridade semântica moderada com o gabarito (SBERTScore F1: {sbert_f1:.2f}).")
    else:
        explicacao.append(f"❌ Baixa similaridade semântica com o gabarito (SBERTScore F1: {sbert_f1:.2f}).")

    # --- LEGISLAÇÃO ---
    if legislacao_result["match"] is None:
        explicacao.append("ℹ️ Nenhuma legislação foi exigida para esta questão.")
    elif legislacao_result["match"]:
        explicacao.append(f"✅ Legislação correta mencionada (Lei {legislacao_result['esperada']} identificada na resposta).")
    else:
        encontrados = legislacao_result.get("encontrados", [])
        if encontrados:
            explicacao.append(f"❌ Legislação incorreta: era esperada a Lei {legislacao_result['esperada']}, mas foram encontradas referências a: {', '.join(encontrados)}. Penalização de 30% aplicada.")
        else:
            explicacao.append(f"❌ Nenhuma legislação foi mencionada na resposta (esperada: Lei {legislacao_result['esperada']}). Penalização de 30% aplicada.")

    # --- NOTA ---
    percentual = (nota_final / valor_questao) * 100
    if percentual >= 85:
        nivel = "Excelente"
    elif percentual >= 65:
        nivel = "Satisfatório"
    elif percentual >= 40:
        nivel = "Parcial"
    else:
        nivel = "Insuficiente"

    explicacao.append(f"\n📝 Nota final: {nota_final}/{valor_questao}.0 ({percentual:.0f}%) — {nivel}.")

    return "\n".join(explicacao)


explicacao = gerar_explicacao(
    sbert_f1=sbertscore_result["f1"],
    nli_result=nli_result,
    legislacao_result=legislacao_result,
    nota_final=final_result,
    valor_questao=1
)

print("\n🔹 Explicação da Nota:")
print(explicacao)

Loading weights:   0%|          | 0/391 [00:00<?, ?it/s]

BertModel LOAD REPORT from: stjiris/bert-large-portuguese-cased-legal-mlm-sts-v1.0
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.
Invalid model-index. Not loading eval results into CardData.


Loading weights:   0%|          | 0/202 [00:00<?, ?it/s]

DebertaV2ForSequenceClassification LOAD REPORT from: MoritzLaurer/mDeBERTa-v3-base-mnli-xnli
Key                             | Status     |  | 
--------------------------------+------------+--+-
deberta.embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

BertModel LOAD REPORT from: bert-base-multilingual-cased
Key                                        | Status     |  | 
-------------------------------------------+------------+--+-
cls.seq_relationship.bias                  | UNEXPECTED |  | 
cls.predictions.transform.LayerNorm.weight | UNEXPECTED |  | 
cls.seq_relationship.weight                | UNEXPECTED |  | 
cls.predictions.transform.dense.weight     | UNEXPECTED |  | 
cls.predictions.transform.dense.bias       | UNEXPECTED |  | 
cls.predictions.bias                       | UNEXPECTED |  | 
cls.predictions.transform.LayerNorm.bias   | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.



📊 RESULTADOS

🔹 BERTScore:
{'precision': 0.9648355841636658, 'recall': 0.9493046402931213, 'f1': 0.957007110118866}

🔹 SBERTScore:
{'precision': 0.8303647637367249, 'recall': 0.8303647637367249, 'f1': 0.8303647637367249}

🔹 Legislação:
{'esperada': '2010', 'esperada_norm': '2010', 'encontrados': ['2010', '2010'], 'match': True}

🔹 NLI:
{'label': 'CONTRADICTION', 'score_bruto': 0.9942428469657898, 'score_convertido': 0.005757153034210205}

🔹 Nota Final:
0.01 / 1.0

🔹 Explicação da Nota:
❌ A resposta contradiz o gabarito (NLI: contradição com score 0.99), o que resultou em penalização severa.
⚠️ Similaridade semântica moderada com o gabarito (SBERTScore F1: 0.83).
✅ Legislação correta mencionada (Lei 2010 identificada na resposta).

📝 Nota final: 0.01/1.0 (1%) — Insuficiente.
